# Notebook 19 — NSV Stage 2: Intrinsic Dimension Estimation  *(CRITICAL RESULT)*
**Project:** ENSO-BSISO SSL — Neural State Variables extension  
**Author:** Jiayi (jh9141@nyu.edu)

**Stage 2 of the NSV pipeline** (Chen et al., §2.2). The Stage 1 encoder from nb18 mapped each daily atmospheric field `X_t ∈ ℝ^{3×31×51}` to an overparameterized latent `z_t ∈ ℝ^{64}`. By the manifold hypothesis, the resulting point cloud `{z_t}` should lie on a lower-dimensional manifold whose dimension equals the true number of state variables of the underlying dynamical system. This notebook estimates that dimension.

**Why this is the headline notebook.** The estimated `d̂` directly answers our central scientific question:
- `d̂ ≈ 2` → BSISO is fully described by 2 state variables (phase + amplitude). ENSO modulates *within* the same 2-D manifold — Hypothesis **H1**.
- `d̂ ≈ 3` → BSISO needs 3 state variables. The 3rd dimension is likely ENSO state — Hypothesis **H2** (the most scientifically interesting outcome).
- `d̂ ≥ 4` → BSISO is more complex than the conventional index suggests — Hypothesis **H3**.

## Method (per Chen et al., §2.2)

**Levina-Bickel (2004) maximum-likelihood ID estimator.** For each point `z^(i)`, compute Euclidean distances to its `k` nearest neighbors: `T_1^(i) ≤ T_2^(i) ≤ … ≤ T_k^(i)`. The local ID estimate is:

$$\hat{m}_k(z^{(i)}) = \left[\,\frac{1}{k-1} \sum_{j=1}^{k-1} \log\frac{T_k^{(i)}}{T_j^{(i)}}\,\right]^{-1}$$

and the global estimate is the average over all `i`. Crucially the formula uses **log ratios of distances**, which is scale-invariant — the absolute magnitude of `z` doesn't affect the ID estimate.

We sweep `k` over `int(N × c)` for `c ∈ {0.008, 0.010, 0.012, 0.014, 0.016}` (5 values) — the exact range Chen et al. use. Mean ± std of the 5 estimates is the headline number.

## Cross-checks (different ID estimators on the same data)

Three estimators that should roughly agree if the result is robust:

| Method | Citation | Idea |
|---|---|---|
| **MLE (Levina-Bickel)** | Levina & Bickel 2004 | k-NN log-distance ratios, max likelihood |
| **Two-NN** | Facco et al. 2017 | uses only the 2 nearest neighbors → most robust to manifold curvature |
| **local PCA** | Fukunaga & Olsen 1971 | per-neighborhood SVD; counts eigenvalues above threshold |

Plus three **sanity controls**:

- **Random Gaussian noise** in ℝ^{64} with the same `N` → should give `d̂ ≈ 64` (full ambient dim). Calibrates the estimator.
- **Per-dimension shuffled** `z_train` → destroys cross-dim correlations while preserving marginal distributions. Should give a much higher `d̂` than the real data if the manifold structure is real.
- **Split by ENSO category** (EN / Neutral / LN) → if the manifold structure is the same across ENSO states (H1), all three groups give similar `d̂`. If ENSO adds a dimension (H2), the within-group `d̂` should be one lower than the global `d̂`.

## Inputs (from nb18)

- `latents/z_train.npy` shape `(5168, 64)` float32
- `latents/z_val.npy` shape `(1368, 64)` float32
- labels from nb17: `bsiso_phase_t.npy`, `bsiso_amplitude_t.npy`, `enso_cat_t.npy`, `train_mask.npy`

## Outputs (`BSISO_SSL_Project/nsv/results/stage2/`)

- `intrinsic_dim.json` — full numerical result + `d_hat` (rounded integer)
- `id_estimation.png` — Levina-Bickel k-sweep + Two-NN + lPCA on one panel
- `controls.png` — real vs random-noise vs shuffled
- `pca_visualization.png` — 2-D PCA of `z`, colored by BSISO phase and by ENSO category
- `stage2_summary.md` — auto-generated decision text

## Decision rule for nb20

After running this notebook the **integer-rounded `d̂`** sets the SIREN bottleneck in nb20:

- `LB_std < 0.5` and all three methods within ±0.5 of each other → confidence high; `d̂ = round(LB_mean)`
- `LB_std > 0.5` → noisy estimate; report both `floor(d̂)` and `ceil(d̂)`, train two SIRENs, compare reconstruction loss
- Methods disagree by > 1 → flag and investigate before nb20

## Runtime

~2–4 min on Colab T4 (skdim is fast at N ≈ 6500).

---

## Cell 1 — Setup: Install `scikit-dimension`, Load Latents + Labels

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# scikit-dimension is the canonical Python lib for ID estimators (Levina-Bickel, Two-NN, lPCA).
# ~30 s install on Colab. Pin a recent version for reproducibility.
!pip install -q scikit-dimension==0.3.4

import os, json
import numpy as np
import matplotlib.pyplot as plt
import skdim

PROJECT_DIR = '/content/drive/MyDrive/BSISO_SSL_Project'
NSV_DIR     = f'{PROJECT_DIR}/nsv'
DATA_DIR    = f'{NSV_DIR}/data'
LATENT_DIR  = f'{NSV_DIR}/latents'
RESULTS_DIR = f'{NSV_DIR}/results/stage2'
os.makedirs(RESULTS_DIR, exist_ok=True)

SEED = 42
rng = np.random.default_rng(SEED)

# Load nb18 latents
z_train = np.load(f'{LATENT_DIR}/z_train.npy')
z_val   = np.load(f'{LATENT_DIR}/z_val.npy')
z_all   = np.concatenate([z_train, z_val], axis=0)

# Load nb17 labels (need ENSO and phase for stratified ID estimates)
train_mask     = np.load(f'{DATA_DIR}/train_mask.npy')
phase_t        = np.load(f'{DATA_DIR}/bsiso_phase_t.npy')
enso_t         = np.load(f'{DATA_DIR}/enso_cat_t.npy')
# Re-order to match z_all = [train..., val...]
order = np.concatenate([np.where(train_mask)[0], np.where(~train_mask)[0]])
phase_all = phase_t[order]
enso_all  = enso_t[order]

# Deduplicate (skdim docs recommend this — duplicates cause T_1 = 0 distances)
_, unique_idx = np.unique(z_all, axis=0, return_index=True)
unique_idx = np.sort(unique_idx)   # preserve order
z = z_all[unique_idx]
phase = phase_all[unique_idx]
enso  = enso_all[unique_idx]
N, D = z.shape

print(f'Loaded latents:  z_train {z_train.shape}, z_val {z_val.shape}, combined unique {z.shape}.')
print(f'Latent stats:    mean ||z||={np.linalg.norm(z, axis=1).mean():.4f},  per-dim std min={z.std(0).min():.4f}, max={z.std(0).max():.4f}.')
assert N > 500, f'Too few unique points ({N}) for stable ID estimation.'
print(f'✓ Deduplication kept {N}/{z_all.shape[0]} ({100*N/z_all.shape[0]:.1f}%) unique points.')

# Centering — not required for scale-invariant ID estimators but conventional
z_centered = z - z.mean(axis=0)

## Cell 2 — Levina-Bickel ID with k-Sweep (Chen et al. Recipe)

Sweep `k ∈ {int(N × 0.008), int(N × 0.010), int(N × 0.012), int(N × 0.014), int(N × 0.016)}`. Report `mean ± std`. Plot ID vs k — a flat curve means the estimate is robust to neighborhood-size choice; a strongly sloped curve suggests the manifold has non-uniform local geometry.

In [ ]:
k_fractions = [0.008, 0.010, 0.012, 0.014, 0.016]
k_list = [max(int(N * c), 3) for c in k_fractions]
print(f'N = {N}  →  k_list = {k_list}  (fractions {k_fractions})')

id_lb_by_k = []
for k in k_list:
    est = skdim.id.MLE(K=k)
    est.fit(z)
    id_lb_by_k.append(float(est.dimension_))
    print(f'  k={k:4d}  →  ID_LB = {id_lb_by_k[-1]:.3f}')

id_lb_by_k = np.asarray(id_lb_by_k)
LB_mean = float(id_lb_by_k.mean())
LB_std  = float(id_lb_by_k.std())
print(f'\nLevina-Bickel:  d̂ = {LB_mean:.3f} ± {LB_std:.3f}  (across 5 k values)')

# Plot the sweep
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_list, id_lb_by_k, 'o-', color='steelblue', lw=2, ms=10, label='Levina-Bickel ID per k')
ax.axhline(LB_mean, color='red', ls='--', lw=1.5, label=f'Mean = {LB_mean:.2f}')
ax.fill_between(k_list, LB_mean - LB_std, LB_mean + LB_std, color='red', alpha=0.15, label=f'±1σ = ±{LB_std:.2f}')
ax.set_xlabel('Neighborhood size k')
ax.set_ylabel('ID estimate')
ax.set_title(f'Levina-Bickel ID sweep over k  (N = {N})', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/id_lb_sweep.png', dpi=140, bbox_inches='tight')
plt.show()

## Cell 3 — Cross-Check Estimators (Two-NN + lPCA) and Sanity Controls

If MLE, Two-NN and lPCA all give similar numbers, the estimate is robust. Then the controls calibrate the estimator: random Gaussian noise in ℝ^{64} should return ~64; per-dim-shuffled latents should return a much higher value than the real data.

In [ ]:
# ── Two-NN (Facco et al. 2017) — uses only 2 nearest neighbors, scale invariant by construction.
est_tnn = skdim.id.TwoNN()
est_tnn.fit(z)
id_tnn = float(est_tnn.dimension_)
print(f'Two-NN:                {id_tnn:.3f}')

# ── local PCA — per-neighborhood SVD, counts eigenvalues above threshold.
est_lpca = skdim.id.lPCA()
est_lpca.fit(z)
id_lpca = float(est_lpca.dimension_)
print(f'local PCA:             {id_lpca:.3f}')

# ── Sanity control A: random Gaussian noise in R^64 of same N.  Should give ID ≈ 64.
z_noise = rng.standard_normal((N, D)).astype(np.float32)
id_noise_tnn = float(skdim.id.TwoNN().fit(z_noise).dimension_)
id_noise_mle = float(skdim.id.MLE(K=k_list[2]).fit(z_noise).dimension_)
print(f'\n[Control A] Gaussian noise in R^{D}, N={N}:')
print(f'    Two-NN = {id_noise_tnn:.2f}   MLE(k={k_list[2]}) = {id_noise_mle:.2f}   (expect ≈ {D})')

# ── Sanity control B: per-dim shuffled real latents (destroys cross-dim correlations).
z_shuffled = z.copy()
for d_i in range(D):
    rng.shuffle(z_shuffled[:, d_i])
id_shuf_tnn = float(skdim.id.TwoNN().fit(z_shuffled).dimension_)
id_shuf_mle = float(skdim.id.MLE(K=k_list[2]).fit(z_shuffled).dimension_)
print(f'\n[Control B] Per-dim shuffled z (real marginals, destroyed correlations):')
print(f'    Two-NN = {id_shuf_tnn:.2f}   MLE(k={k_list[2]}) = {id_shuf_mle:.2f}   (should be > {LB_mean:.1f} if manifold is real)')

# ── Summary table
print('\n' + '=' * 70)
print(f'INTRINSIC DIMENSION SUMMARY  (N={N}, ambient D={D})')
print('=' * 70)
print(f'  Levina-Bickel (k-sweep)    : {LB_mean:.3f} ± {LB_std:.3f}')
print(f'  Two-NN                     : {id_tnn:.3f}')
print(f'  local PCA                  : {id_lpca:.3f}')
print(f'  ── controls ──')
print(f'  Gaussian noise (Two-NN)    : {id_noise_tnn:.3f}   (expect ≈ {D})')
print(f'  Shuffled z (Two-NN)        : {id_shuf_tnn:.3f}   (real data must be lower)')

# Plot: all estimators side by side
fig, ax = plt.subplots(figsize=(11, 5))
labels = ['LB (k-sweep)', 'Two-NN', 'lPCA', 'Gaussian noise\n(Two-NN)', 'Shuffled z\n(Two-NN)']
vals   = [LB_mean,         id_tnn,   id_lpca, id_noise_tnn,                id_shuf_tnn]
errs   = [LB_std,          0,        0,       0,                           0]
colors = ['steelblue',     '#1f77b4','#1f77b4', '#888888',                  '#d62728']
bars = ax.bar(labels, vals, yerr=errs, color=colors, alpha=0.85, capsize=6)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.5, f'{v:.2f}', ha='center', fontsize=11, fontweight='bold')
ax.axhline(D, color='gray', ls=':', lw=1, label=f'Ambient dim D={D}')
ax.set_ylabel('ID estimate')
ax.set_title('ID estimators on real latents vs sanity controls', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/controls.png', dpi=140, bbox_inches='tight')
plt.show()

## Cell 4 — Stratified ID by ENSO Category (H1 vs H2 Diagnostic)

If ENSO adds an independent state-space dimension (H2), then *within* one ENSO category (e.g., "all El Niño days") the manifold should be `(d̂ − 1)`-dimensional — because the ENSO axis is now held constant. If ENSO merely modulates BSISO within the same manifold (H1), then within-category ID should equal global ID.

We compute ID separately for {EN, Neutral, LN} subsets and compare to the global estimate.

In [ ]:
id_by_enso = {}
for cat in ['El Nino', 'Neutral', 'La Nina']:
    mask = (enso == cat)
    n_cat = int(mask.sum())
    if n_cat < 200:
        id_by_enso[cat] = {'n': n_cat, 'tnn': float('nan'), 'mle': float('nan')}
        print(f'  {cat:9s}:  N = {n_cat:5d}  (too few for ID — skipped)')
        continue
    z_cat = z[mask]
    k_cat = max(int(n_cat * 0.012), 3)
    tnn = float(skdim.id.TwoNN().fit(z_cat).dimension_)
    mle = float(skdim.id.MLE(K=k_cat).fit(z_cat).dimension_)
    id_by_enso[cat] = {'n': n_cat, 'tnn': tnn, 'mle': mle, 'k': k_cat}
    print(f'  {cat:9s}:  N = {n_cat:5d}  Two-NN = {tnn:.2f}  MLE(k={k_cat}) = {mle:.2f}')

global_tnn = id_tnn
global_mle = LB_mean

print('\nInterpretation (Two-NN-based):')
print(f'  Global ID:        {global_tnn:.2f}')
for cat, r in id_by_enso.items():
    if not np.isnan(r['tnn']):
        delta = r['tnn'] - global_tnn
        flag = '(≈ global → H1 consistent)' if abs(delta) < 0.5 else f'(Δ = {delta:+.2f} → H2 consistent if Δ ≈ −1)'
        print(f'  {cat:9s} ID:     {r["tnn"]:.2f}  {flag}')

# Visualize
fig, ax = plt.subplots(figsize=(9, 5))
cats = ['El Nino', 'Neutral', 'La Nina']
x = np.arange(len(cats) + 1)
tnn_vals = [global_tnn] + [id_by_enso[c]['tnn'] for c in cats]
mle_vals = [global_mle] + [id_by_enso[c]['mle'] for c in cats]
ns       = [N]          + [id_by_enso[c]['n']   for c in cats]
labels   = ['Global'] + cats
w = 0.35
ax.bar(x - w/2, tnn_vals, width=w, color='#1f77b4', alpha=0.85, label='Two-NN')
ax.bar(x + w/2, mle_vals, width=w, color='steelblue', alpha=0.85, label='MLE (Levina-Bickel)')
for xi, (t, m, n) in enumerate(zip(tnn_vals, mle_vals, ns)):
    if not np.isnan(t): ax.text(xi - w/2, t + 0.10, f'{t:.2f}', ha='center', fontsize=10)
    if not np.isnan(m): ax.text(xi + w/2, m + 0.10, f'{m:.2f}', ha='center', fontsize=10)
    ax.text(xi, -0.5, f'N={n}', ha='center', fontsize=9, color='gray')
ax.axhline(global_tnn, color='gray', ls='--', lw=1, alpha=0.5)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('ID estimate')
ax.set_title('ID stratified by ENSO category  (H1: equal across cats;  H2: within-cat is 1 lower than global)',
             fontweight='bold', fontsize=11)
ax.legend(); ax.grid(alpha=0.3, axis='y')
ax.set_ylim(bottom=-1)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/id_by_enso.png', dpi=140, bbox_inches='tight')
plt.show()

## Cell 5 — 2-D Visualization of the Latent Manifold

If `d̂ ≈ 2`, a 2-D PCA of the 64-D latent should show **almost all** the variance (> 90%) and **most of the manifold structure** visually. If `d̂ ≥ 3`, the first two PCs capture less variance and additional structure lives in higher PCs.

Color by BSISO phase (expect circular organization 1 → 2 → … → 8 → 1) and by ENSO (expect separated clusters if H2, mixed if H1).

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=min(10, D)).fit(z_centered)
z_pca = pca.transform(z_centered)
var_ratio = pca.explained_variance_ratio_
cum_var   = np.cumsum(var_ratio)
print(f'Variance explained by first PCs:')
for i in range(min(6, len(var_ratio))):
    print(f'  PC{i+1}: {var_ratio[i]*100:5.2f}%   (cum {cum_var[i]*100:5.2f}%)')

fig = plt.figure(figsize=(18, 5))

# Panel 1: variance scree
ax = fig.add_subplot(1, 3, 1)
ax.bar(range(1, len(var_ratio)+1), var_ratio*100, color='steelblue', alpha=0.85)
ax.set_xlabel('PC index'); ax.set_ylabel('% variance explained')
ax.set_title(f'Scree (first 10 PCs)\nCum at d̂ ≈ {round(LB_mean)}: {cum_var[max(0, round(LB_mean)-1)]*100:.1f}%',
             fontweight='bold', fontsize=11)
ax.grid(alpha=0.3, axis='y')

# Panel 2: PCA 2-D scatter colored by BSISO phase
ax = fig.add_subplot(1, 3, 2)
phase_colors = plt.cm.hsv(np.linspace(0, 1, 9))[:8]   # cyclic palette
for p in range(1, 9):
    m = phase == p
    ax.scatter(z_pca[m, 0], z_pca[m, 1], c=[phase_colors[p-1]], s=8, alpha=0.6, label=f'P{p}')
ax.set_xlabel(f'PC1 ({var_ratio[0]*100:.1f}%)'); ax.set_ylabel(f'PC2 ({var_ratio[1]*100:.1f}%)')
ax.set_title('Latent manifold colored by BSISO phase\n(expect ring 1→2→…→8→1 if d̂=2)', fontweight='bold', fontsize=11)
ax.legend(fontsize=8, ncol=2, loc='best'); ax.grid(alpha=0.3); ax.set_aspect('equal')

# Panel 3: PCA 2-D scatter colored by ENSO
ax = fig.add_subplot(1, 3, 3)
enso_cmap = {'El Nino': '#d62728', 'Neutral': '#7f7f7f', 'La Nina': '#1f77b4'}
enso_marker = {'El Nino': '^', 'Neutral': 'o', 'La Nina': 's'}
for cat in ['El Nino', 'Neutral', 'La Nina']:
    m = enso == cat
    ax.scatter(z_pca[m, 0], z_pca[m, 1], c=enso_cmap[cat], marker=enso_marker[cat],
               s=8, alpha=0.55, label=f'{cat} (N={int(m.sum())})')
ax.set_xlabel(f'PC1 ({var_ratio[0]*100:.1f}%)'); ax.set_ylabel(f'PC2 ({var_ratio[1]*100:.1f}%)')
ax.set_title('Latent manifold colored by ENSO\n(H2: clear separation; H1: well-mixed)', fontweight='bold', fontsize=11)
ax.legend(fontsize=9, loc='best'); ax.grid(alpha=0.3); ax.set_aspect('equal')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/pca_visualization.png', dpi=140, bbox_inches='tight')
plt.show()

# Diagnostic number: cumulative variance at integer-rounded d̂
d_hat = max(1, round(LB_mean))
cum_at_dhat = float(cum_var[d_hat - 1])
print(f'\nCumulative variance captured by first {d_hat} PCs: {cum_at_dhat*100:.1f}%')
print(f'  → if d̂ is the true ID, this should be high (>80%); otherwise more PCs are needed.')

## Cell 6 — Final Decision: Save `intrinsic_dim.json` and Write Summary

Apply the Session 26 decision rule, save the headline number for nb20, and write a markdown summary.

In [ ]:
import time as _t

# Decision logic (Session 26 plan §6)
methods_close = (abs(id_tnn - LB_mean) <= 0.5) and (abs(id_lpca - LB_mean) <= 0.5)
lb_tight      = LB_std < 0.5
manifold_real = id_shuf_tnn > LB_mean + 0.5   # shuffled control sanity
noise_calib   = id_noise_tnn > D * 0.7         # estimator calibrated

d_hat = max(1, round(LB_mean))

if not noise_calib:
    confidence = 'LOW'
    decision_text = (f'**Estimator calibration failed**: Gaussian-noise control returned {id_noise_tnn:.1f} '
                     f'(expected ≈ {D}). Something is off in the skdim install or numerical handling. '
                     f'Re-verify before trusting d̂.')
elif not manifold_real:
    confidence = 'LOW'
    decision_text = (f'**Manifold reality check failed**: shuffled-z gave d̂ = {id_shuf_tnn:.1f}, '
                     f'only marginally larger than the real-data d̂ = {LB_mean:.1f}. '
                     f'The cross-dimensional correlations in z may be too weak to assert manifold structure. '
                     f'Consider retraining Stage 1 with more epochs or a different LR.')
elif lb_tight and methods_close:
    confidence = 'HIGH'
    decision_text = (f'**High-confidence ID estimate: d̂ = {d_hat}.** '
                     f'Levina-Bickel ({LB_mean:.2f} ± {LB_std:.2f}), Two-NN ({id_tnn:.2f}), '
                     f'and lPCA ({id_lpca:.2f}) all agree within ±0.5. '
                     f'Sanity controls passed (noise → {id_noise_tnn:.1f} ≈ {D}; shuffled → {id_shuf_tnn:.1f} ≫ d̂). '
                     f'Proceed to nb20 with SIREN bottleneck = {d_hat}.')
elif lb_tight:
    confidence = 'MEDIUM'
    decision_text = (f'**Medium-confidence ID estimate: d̂ = {d_hat}.** '
                     f'Levina-Bickel ({LB_mean:.2f}) is tight across k, but Two-NN ({id_tnn:.2f}) '
                     f'and lPCA ({id_lpca:.2f}) differ by > 0.5. '
                     f'Proceed to nb20 with bottleneck = {d_hat}, but also try {d_hat-1 if d_hat>1 else 1} and {d_hat+1} as ablations.')
else:
    confidence = 'LOW'
    other = sorted({max(1, int(np.floor(LB_mean))), max(1, int(np.ceil(LB_mean)))})
    decision_text = (f'**Low-confidence ID estimate**: LB std {LB_std:.2f} > 0.5 across k. '
                     f'Train two SIRENs in nb20 with bottleneck values {other} and compare '
                     f'reconstruction loss + downstream BSISO-phase organization.')

# Scientific interpretation
if d_hat == 2:
    interp = ('Hypothesis **H1** confirmed: BSISO is intrinsically 2-D. ENSO modulates within the same 2-D phase manifold '
              'rather than adding a state variable. The 3rd ENSO axis we hoped to find is absent in the dynamical state space.')
elif d_hat == 3:
    interp = ('Hypothesis **H2** confirmed: BSISO needs 3 state variables. The 3rd dimension is the candidate ENSO axis — '
              'nb20 will verify by training a SIREN with bottleneck = 3 and checking whether the 3rd dim correlates with '
              'Niño 3.4. This would be the strongest scientific result of the project: ENSO occupies an independent degree '
              'of freedom in BSISO state space, not just a modulation within the BSISO manifold.')
elif d_hat >= 4:
    interp = (f'Hypothesis **H3** territory: BSISO state requires ≥ {d_hat} dimensions. Beyond ENSO, additional structure '
              f'lives in the latent. Could be Indian Ocean SST, monsoon trough latitude, or slower modes. Worth investigating '
              f'in nb20 whether the extra dimensions have physical correlates.')
else:
    interp = f'd̂ = {d_hat} is unexpectedly low. BSISO has at least phase + amplitude → ID ≥ 2 expected. Inspect Stage 1 reconstructions.'

# Save full JSON
result = {
    'date':                _t.strftime('%Y-%m-%d'),
    'n_samples':           int(N),
    'ambient_dim':         int(D),
    'k_list':              [int(k) for k in k_list],
    'k_fractions':         k_fractions,
    'LB_by_k':             [float(v) for v in id_lb_by_k],
    'LB_mean':             float(LB_mean),
    'LB_std':              float(LB_std),
    'TwoNN':               float(id_tnn),
    'lPCA':                float(id_lpca),
    'd_hat':               int(d_hat),
    'confidence':          confidence,
    'controls': {
        'noise_TwoNN':     float(id_noise_tnn),
        'noise_MLE':       float(id_noise_mle),
        'shuffled_TwoNN':  float(id_shuf_tnn),
        'shuffled_MLE':    float(id_shuf_mle),
    },
    'id_by_enso':          id_by_enso,
    'pca_cumvar_at_dhat':  float(cum_at_dhat),
}
with open(f'{RESULTS_DIR}/intrinsic_dim.json', 'w') as f:
    json.dump(result, f, indent=2, default=float)

# Save markdown summary
summary_md = f"""# NSV Stage 2 — Intrinsic Dimension Estimate

**Date:** {_t.strftime('%Y-%m-%d')}  
**Input:** {N} deduplicated 64-D latent vectors from nb18's Stage 1 encoder (BSISO MJJAS).

## Headline number

**d̂ = {d_hat}**  ({confidence} confidence)

| Method | Estimate |
|---|---|
| Levina-Bickel (k-sweep, mean ± std) | **{LB_mean:.3f} ± {LB_std:.3f}** |
| Two-NN (Facco et al. 2017) | {id_tnn:.3f} |
| local PCA | {id_lpca:.3f} |

## Sanity controls

| Control | Two-NN | MLE(k={k_list[2]}) | Verdict |
|---|---|---|---|
| Gaussian noise in R^{D}, same N | {id_noise_tnn:.2f} | {id_noise_mle:.2f} | {'✓' if noise_calib else '✗'} estimator calibrated (expect ≈ {D}) |
| Shuffled z (real marginals, no cross-dim) | {id_shuf_tnn:.2f} | {id_shuf_mle:.2f} | {'✓' if manifold_real else '✗'} manifold is real (shuffled ID ≫ real ID) |

## Stratified by ENSO

| Subset | N | Two-NN | MLE | Δ from global Two-NN |
|---|---|---|---|---|
{chr(10).join(f'| {cat:9s} | {r["n"]:5d} | {r["tnn"]:.2f} | {r["mle"]:.2f} | {r["tnn"] - global_tnn:+.2f} |' for cat, r in id_by_enso.items() if not np.isnan(r['tnn']))}

**H2 signature** would be: within-category ID ≈ global − 1 (because fixing ENSO removes one state-space axis).  
**H1 signature** would be: within-category ID ≈ global (ENSO modulates within the same manifold).

## Decision

{decision_text}

## Scientific interpretation

{interp}

## Files

```
results/stage2/intrinsic_dim.json     ← headline JSON (consumed by nb20)
results/stage2/id_lb_sweep.png        ← Levina-Bickel k-sweep
results/stage2/controls.png           ← all estimators + sanity controls
results/stage2/id_by_enso.png         ← ENSO-stratified ID
results/stage2/pca_visualization.png  ← 2-D PCA scatter (BSISO phase + ENSO)
results/stage2/stage2_summary.md      ← this file
```
"""

with open(f'{RESULTS_DIR}/stage2_summary.md', 'w') as f:
    f.write(summary_md)

# Final console block
print('=' * 78)
print(f'  FINAL d̂ = {d_hat}  ({confidence} confidence)')
print('=' * 78)
print(decision_text)
print()
print('Scientific interpretation:')
print(interp)
print()
print(f'Saved: {RESULTS_DIR}/intrinsic_dim.json')
print(f'Saved: {RESULTS_DIR}/stage2_summary.md')
print('\n→ nb20 reads intrinsic_dim.json to set the SIREN refine bottleneck.')

---
## Done!

**Send back** for review:
1. The final console block from Cell 6 — `d̂` + confidence + interpretation.
2. `results/stage2/controls.png` — all 5 estimators on one bar chart. The shuffled control should be much higher than the real-data estimates; the noise control should be near 64.
3. `results/stage2/pca_visualization.png` — most informative single figure. If the BSISO-phase panel shows a clean ring, d̂ ≈ 2 is real. If the ENSO panel shows separated EN/LN clusters, ENSO adds a dimension.
4. `results/stage2/id_by_enso.png` — the H1-vs-H2 verdict figure.

**Once we have a d̂, nb20 does the last three stages of NSV:**
- **Stage 3**: SIREN autoencoder on the 64-D latent with bottleneck = d̂ → produces Neural State Variables `v_t ∈ ℝ^{d̂}`.
- **Stage 4**: small MLP dynamics predictor `f: v_t → v_{t+1}` to verify the discovered state variables support next-step prediction.
- **Analysis**: correlations of `v_t` with BSISO PC1/PC2; ENSO displacement z-score in v-space; long-term rollout stability.

---
*DDCS Project | jh9141@nyu.edu*